# Bonus: Querying Overture Maps with Ibis

This notebook extends the DuckDB work from Exercises 01–03 by introducing **Ibis** — a Python dataframe API that compiles to DuckDB SQL under the hood.

## Why Ibis?

In the earlier exercises we wrote raw SQL strings. Ibis lets you express the same logic using **Python method chaining** — similar to pandas — and then transpiles that to DuckDB SQL. This means:

- Auto-complete and type checking in your IDE
- The same Python code runs against other backends (BigQuery, Snowflake, Spark) with zero changes
- Lazy evaluation: the query isn't sent to DuckDB until you actually request results

## What We'll Query

The **Overture Maps `base/infrastructure`** theme — specifically power infrastructure across the USA (power plants, towers, lines, substations). This is the same Overture S3 bucket from Exercise 01, just a different theme.

---

## Setup

Install the packages if you haven't already:

```bash
uv add 'ibis-framework[duckdb]' lonboard geopandas pyarrow
```

In [ ]:
import ibis
from ibis import _

# Connect to a local DuckDB instance (in-memory by default)
con = ibis.duckdb.connect()

# Enable the S3/httpfs extension so DuckDB can read from Overture's public bucket
con.raw_sql("INSTALL httpfs; LOAD httpfs;")
con.raw_sql("INSTALL spatial; LOAD spatial;")
con.raw_sql("SET s3_region='us-west-2';")

# Make ibis show results immediately (interactive mode)
ibis.options.interactive = True

print("Ibis version:", ibis.__version__)
print("Connected to:", con)

## Exercise 1 — Explore the Infrastructure Schema

The `base/infrastructure` theme contains things like power lines, towers, substations, generators — the physical backbone of the electrical grid. Let's query it directly from Overture's public S3 bucket using Ibis's `read_parquet()` method.

In [ ]:
# Point Ibis at the infrastructure theme on Overture's public S3 bucket
# The glob pattern * picks up all hive-partitioned parquet shards
url = "s3://overturemaps-us-west-2/release/2024-07-22.0/theme=base/type=infrastructure/*"

t = con.read_parquet(url, table_name="infra_global")

# What columns do we have?
print(t.schema())

### What's in the `class` column?

The `class` field tags what kind of infrastructure each record represents. Let's see what categories exist and how many records fall into each one.

> 📖 **Ibis note**: `.value_counts()` compiles to `GROUP BY ... COUNT(*)` with an ORDER BY. Notice we don't write any SQL — Ibis generates it for us.

In [ ]:
# Count records by infrastructure class, sorted descending
(
    t.filter(
        # Filter to USA continental bounding box to shrink the dataset
        _.bbox.xmin > -125.0,
        _.bbox.ymin > 24.8,
        _.bbox.xmax < -65.8,
        _.bbox.ymax < 49.2,
    )
    .rename(infra_class="class")
    .infra_class
    .value_counts()
    .order_by(ibis.desc("infra_class_count"))
    .limit(20)
)

## Exercise 2 — Filter to USA + Subtype

Now let's apply multiple filters using Ibis's method chain. We'll focus on **power infrastructure** only, within the USA continental bounding box.

In [ ]:
# Build a filtered expression (this is lazy — no data is fetched yet)
usa_power_expr = (
    t.filter(
        _.bbox.xmin > -125.0,
        _.bbox.ymin > 24.8,
        _.bbox.xmax < -65.8,
        _.bbox.ymax < 49.2,
        _.subtype == "power",
    )
    .select(["names", "geometry", "bbox", "class", "sources", "source_tags"])
    .rename(infra_class="class")
)

# Show what SQL Ibis will execute (great for debugging!)
print(ibis.to_sql(usa_power_expr))

In [ ]:
# Now materialize to a local GeoParquet file
# This is the expensive step — DuckDB streams from S3 and filters in-memory
con.to_parquet(usa_power_expr, "power-infra-usa.geoparquet")
print("Saved to power-infra-usa.geoparquet")

In [ ]:
# Re-read the local file for faster subsequent queries
usa_power = con.read_parquet("power-infra-usa.geoparquet", table_name="usa_power")
usa_power.count()

## Exercise 3 — Ibis Filtering vs. SQL

Here's an important pattern to understand: **Ibis is just SQL in a Python costume.** Let's prove it by filtering to power plants and comparing Ibis vs. equivalent raw SQL.

> 📖 The `_.infra_class == "plant"` syntax uses Ibis's deferred expression (`_`) to reference columns without quoting them.

In [ ]:
# Ibis style
plants_ibis = usa_power.filter(_.infra_class == "plant").select(_.names.primary, _.geometry, _.bbox)

# Equivalent raw SQL for comparison
print("=== Ibis generates this SQL ===")
print(ibis.to_sql(plants_ibis))

In [ ]:
# Show the first 5 plant records
plants_ibis.limit(5)

## Exercise 4 — Regional Filter and Visualization with lonboard

Let's zoom in on California (by bounding box) and visualize the power plants with **lonboard** — a GPU-accelerated map renderer for Jupyter that handles millions of geometries efficiently.

This is where ibis + lonboard shines compared to Folium: Folium sends all GeoJSON to the browser via JavaScript; lonboard uses **Apache Arrow + WebGL** for near-instant rendering of large datasets.

In [ ]:
import geopandas as gpd
import lonboard
from lonboard.basemap import CartoBasemap

# Filter to LA Metro area bounding box using Ibis
plants_ca = (
    usa_power
    .filter(
        _.infra_class == "plant",
        _.bbox.xmin.between(-118.6, -117.9),
        _.bbox.ymin.between(34.5, 35.3),
    )
    .select(_.names.primary, _.geometry)
)

# Convert to GeoDataFrame (Ibis → pandas → geopandas)
df_ca = plants_ca.to_pandas()
gdf_ca = gpd.GeoDataFrame(
    df_ca,
    geometry=gpd.GeoSeries.from_wkb(df_ca["geometry"]),
    crs="EPSG:4326"
)

# Visualize with lonboard
lonboard.viz(
    gdf_ca,
    scatterplot_kwargs={"get_fill_color": [220, 40, 40, 200], "get_radius": 500},
    polygon_kwargs={"get_fill_color": [220, 40, 40, 180]},
    map_kwargs={
        "basemap_style": CartoBasemap.Positron,
        "view_state": {"longitude": -118.25, "latitude": 34.9, "zoom": 9},
    },
)

## Exercise 5 — Power Lines and Multi-Layer Visualization

lonboard can render multiple GeoPandas layers at once. Let's add transmission lines (major power lines) and minor lines on top of the plant dots.

In [ ]:
# Filter to power lines and minor lines
power_lines = usa_power.filter(_.infra_class == "power_line").select(_.geometry)
minor_lines = usa_power.filter(_.infra_class == "minor_line").select(_.geometry)

# Ibis → pandas → GeoPandas
def to_gdf(expr):
    df = expr.to_pandas()
    return gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkb(df["geometry"]), crs="EPSG:4326")

gdf_power = to_gdf(power_lines)
gdf_minor = to_gdf(minor_lines)

print(f"Power lines: {len(gdf_power):,} features")
print(f"Minor lines: {len(gdf_minor):,} features")

In [ ]:
# Two-layer visualization: power lines (orange) + minor lines (yellow)
lonboard.viz(
    [gdf_power, gdf_minor],
    path_kwargs={"get_color": [255, 140, 0, 180], "width_min_pixels": 1},
    map_kwargs={"basemap_style": CartoBasemap.DarkMatter},
)

## Summary: Ibis vs. Raw SQL vs. GeoPandas

| Approach | Best For | Notes |
|---|---|---|
| **Raw DuckDB SQL** | Quick one-off queries, scripts | What we used in Exercises 01-03 |
| **Ibis** | Python-native analysis, IDE autocomplete, multi-backend portability | Same SQL under the hood |
| **GeoPandas** | Geometry manipulation, spatial operations on moderate data | See Module 34 |
| **lonboard** | Fast visualization of large geodatasets in Jupyter | Uses WebGL/Arrow, much faster than Folium for >10k features |

### Cross-Platform Reference

The Ibis queries in this notebook generate DuckDB SQL that has very close equivalents in:
- **PostGIS**: Replace `ST_GeomFromWKB` → same, add `USING GIST` index hints
- **BigQuery**: Use `ST_GEOGFROMWKB()`, `ST_WITHIN()`, `GEOGRAPHY` type
- **Wherobots/Sedona**: Replace DuckDB with `spark.sql()` + `ST_GeomFromWKB()` in Spark SQL
- **Snowflake**: Use `TO_GEOGRAPHY()` and Snowflake spatial functions

The spatial function names change slightly per platform, but the logical pattern (load → filter by bbox → spatial predicate → aggregate → visualize) is universal.